# Esperimento ontologico 01 — GRU globale vs mosaico causale

Una sola ipotesi nuova: **le 17 transizioni possono essere fattorizzate secondo entità causali locali**. Entrambi i modelli usano esclusivamente il layer pubblicato `torch.nn.GRU`; non introduciamo nuove celle dinamiche. Il controllo è una GRU globale con un budget di parametri quasi uguale.

Riferimento del layer: Cho et al. (2014), *Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation*.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile

def project_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    work = Path('/kaggle/working')
    if work.exists():
        candidates += [p.parent for p in work.glob('*/pyproject.toml')]
    for candidate in candidates:
        marker = candidate / 'pyproject.toml'
        if marker.exists() and 'hay-single-compartment' in marker.read_text():
            return candidate
    destination = Path(tempfile.mkdtemp(prefix='hay_ontology_01_', dir='/kaggle/working'))
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(destination)])
    return destination

ROOT = project_root()
SRC = ROOT / 'src'
assert (SRC / 'hay_single_compartment').is_dir(), f'Package source missing: {SRC}'
sys.path.insert(0, str(SRC))
print('Project:', ROOT)
print('Source:', SRC)

In [ ]:
import h5py
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from hay_single_compartment import (
    INPUT_NAMES, ONTOLOGY_GROUPS, STATE_NAMES, SimulationConfig,
    generate_dataset, validate_dataset,
)
from hay_single_compartment.dataset import Normalization
from hay_single_compartment.models import build_model
from hay_single_compartment.training import rollout_batch, train_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('/kaggle/working/hay_ontology_experiment_01') if Path('/kaggle').exists() else ROOT / 'artifacts' / 'ontology_01'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET = OUTPUT_DIR / 'single_compartment_ontology_v1.h5'
print('Device:', DEVICE, '| output:', OUTPUT_DIR)

## 1. Dataset fisso e tracciato

Le frazioni di dati selezionano intere traiettorie, mai finestre sparse della stessa traiettoria. Validation e test restano invariati.

In [ ]:
config = SimulationConfig(
    duration_ms=500.0, warmup_ms=150.0, seed=27182,
    train_trajectories=24, validation_trajectories=4, test_trajectories=6,
)
dataset_report = generate_dataset(DATASET, config, progress=True) if not DATASET.exists() else validate_dataset(DATASET)
dataset_report

## 2. Ontologia eseguibile

La tabella mostra ciò che ogni GRU locale può leggere e ciò che deve predire. Le dipendenze sono maschere di connettività, non nuovi layer.

In [ ]:
ontology_table = pd.DataFrame([{
    'entity': group.name,
    'outputs': ', '.join(group.state_names),
    'inputs': ', '.join(group.dependency_names),
} for group in ONTOLOGY_GROUPS])
ontology_table.to_csv(OUTPUT_DIR / 'ontology_schema.csv', index=False)
(OUTPUT_DIR / 'ontology_schema.json').write_text(json.dumps([{
    'entity': group.name, 'outputs': list(group.state_names), 'inputs': list(group.dependency_names)
} for group in ONTOLOGY_GROUPS], indent=2), encoding='utf-8')
ontology_table

In [ ]:
MODEL_SETTINGS = {
    'global_gru': dict(architecture='gru', hidden_dim=128, layers=2),
    'ontology_gru': dict(architecture='ontology_gru', hidden_dim=40, layers=2),
}
for name, settings in MODEL_SETTINGS.items():
    probe = build_model(settings['architecture'], len(STATE_NAMES) + len(INPUT_NAMES), len(STATE_NAMES), hidden_dim=settings['hidden_dim'], layers=settings['layers'])
    settings['parameters'] = sum(parameter.numel() for parameter in probe.parameters())
pd.DataFrame(MODEL_SETTINGS).T[['architecture', 'hidden_dim', 'layers', 'parameters']]

## 3. Curva di efficienza dei dati

Si addestrano entrambi i modelli sul 25%, 50% e 100% delle traiettorie. Ogni epoca stampa loss, tempo ed ETA. Il confronto principale non è un singolo record finale, ma l'intera curva errore–dati.

In [ ]:
DATA_FRACTIONS = (0.25, 0.50, 1.00)
reports = []
for fraction in DATA_FRACTIONS:
    for model_name, settings in MODEL_SETTINGS.items():
        run_name = f'{model_name}_data_{int(100 * fraction):03d}'
        print('\n' + '=' * 90)
        print(f'Training {run_name}: {settings["parameters"]:,} parameters, {fraction:.0%} data')
        reports.append(train_model(
            DATASET, OUTPUT_DIR / 'models', settings['architecture'],
            run_name=run_name, train_fraction=fraction, epochs=30,
            sequence_length=128, stride=32, batch_size=32,
            hidden_dim=settings['hidden_dim'], layers=settings['layers'],
            learning_rate=6e-4, dropout=0.1, patience=7, minimum_epochs=15,
            device=DEVICE, seed=27182, use_amp=True, verbose=True,
        ))
print('All controlled runs completed.')

In [ ]:
comparison = pd.DataFrame([{
    'run': report['run_name'], 'model': report['run_name'].split('_data_')[0],
    'data_fraction': report['train_fraction'],
    'train_trajectories': report['train_trajectories'],
    'parameters': report['parameters'], 'epochs': report['epochs_trained'],
    'validation_loss': report['best_validation_loss'],
    'test_voltage_rmse_mV': report['test']['voltage_rmse_mv'],
    'test_normalized_rmse': report['test']['mean_normalized_rmse'],
} for report in reports]).sort_values(['data_fraction', 'validation_loss'])
comparison.to_csv(OUTPUT_DIR / 'data_efficiency_comparison.csv', index=False)
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for model_name, frame in comparison.groupby('model'):
    frame = frame.sort_values('train_trajectories')
    axes[0].plot(frame.train_trajectories, frame.test_voltage_rmse_mV, marker='o', label=model_name)
    axes[1].plot(frame.train_trajectories, frame.test_normalized_rmse, marker='o', label=model_name)
axes[0].set(title='Data efficiency: voltage', xlabel='training trajectories', ylabel='test RMSE (mV)')
axes[1].set(title='Data efficiency: all states', xlabel='training trajectories', ylabel='mean normalized RMSE')
for axis in axes: axis.grid(alpha=.25); axis.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'data_efficiency.png', dpi=160)

## 4. Diagnosi per entità e rollout

Confrontiamo i checkpoint a dati completi. La selezione resta basata sulla validation; il test non sceglie il vincitore.

In [ ]:
full_reports = [report for report in reports if report['train_fraction'] == 1.0]
group_rows = []
for report in full_reports:
    for entity, error in report['test']['per_group_normalized_rmse'].items():
        group_rows.append({'model': report['run_name'].split('_data_')[0], 'entity': entity, 'normalized_rmse': error})
group_comparison = pd.DataFrame(group_rows)
group_comparison.to_csv(OUTPUT_DIR / 'entity_errors.csv', index=False)
group_comparison.pivot(index='entity', columns='model', values='normalized_rmse').sort_values('ontology_gru', ascending=False)

In [ ]:
with h5py.File(DATASET, 'r') as h5:
    truth = h5['test/states'][...]
    future_inputs = h5['test/inputs'][...]
rollout_rows, rollout_predictions = [], {}
HORIZONS_MS = (50, 100, 200, 500)
for report in full_reports:
    model_name = report['run_name'].split('_data_')[0]
    checkpoint = torch.load(report['checkpoint'], map_location=DEVICE, weights_only=False)
    model = build_model(checkpoint['architecture'], len(STATE_NAMES) + len(INPUT_NAMES), len(STATE_NAMES), **checkpoint['model_kwargs']).to(DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    normalization = Normalization.from_dict(checkpoint['normalization'])
    print(f'\nRollout {model_name} on {len(truth)} complete test trajectories...')
    prediction = rollout_batch(model, truth[:, 0], future_inputs, normalization, DEVICE, progress=True)
    rollout_predictions[model_name] = prediction
    for horizon in HORIZONS_MS:
        end = int(horizon / config.dt_ms) + 1
        error = prediction[:, :end] - truth[:, :end]
        rollout_rows.append({
            'model': model_name, 'horizon_ms': horizon,
            'voltage_rmse_mV': float(np.sqrt(np.mean(error[..., 0] ** 2))),
            'mean_normalized_rmse': float(np.sqrt(np.mean((error / normalization.state_std) ** 2, axis=(0, 1))).mean()),
        })
rollout_table = pd.DataFrame(rollout_rows)
rollout_table.to_csv(OUTPUT_DIR / 'rollout_comparison.csv', index=False)
rollout_table

In [ ]:
time_ms = np.arange(truth.shape[1]) * config.dt_ms
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
axes[0].plot(time_ms, truth[0, :, 0], label='teacher', color='black', lw=1.2)
axes[1].plot(time_ms, truth[0, :, 1] * 1e3, label='teacher', color='black', lw=1.2)
for model_name, prediction in rollout_predictions.items():
    axes[0].plot(time_ms, prediction[0, :, 0], label=model_name, alpha=.8)
    axes[1].plot(time_ms, prediction[0, :, 1] * 1e3, label=model_name, alpha=.8)
axes[0].set(ylabel='V (mV)', title='Autoregressive rollout')
axes[1].set(xlabel='time (ms)', ylabel='Ca i (uM)')
for axis in axes: axis.legend(); axis.grid(alpha=.2)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'rollout_example.png', dpi=160)

## Criterio decisionale

L'ipotesi ontologica è supportata soltanto se il mosaico migliora la curva di efficienza a parità approssimativa di parametri e non degrada sistematicamente i rollout. Se fallisce, non si introduce una cella nuova: si analizzano prima le entità responsabili e le dipendenze mancanti.

In [ ]:
from shutil import copytree, make_archive, rmtree
import base64, os
from IPython.display import Javascript, display

include_checkpoints = os.environ.get('HAY_DOWNLOAD_CHECKPOINTS', '0') == '1'
archive_source = OUTPUT_DIR
staging = Path('/kaggle/working/hay_ontology_01_download')
if not include_checkpoints:
    if staging.exists(): rmtree(staging)
    copytree(OUTPUT_DIR, staging, ignore=lambda path, names: {'models'} if 'models' in names else set())
    archive_source = staging
zip_base = Path('/kaggle/working/hay_ontology_experiment_01_complete')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=archive_source.parent, base_dir=archive_source.name))
encoded = base64.b64encode(zip_path.read_bytes()).decode('ascii')
display(Javascript(f"""
const binary = atob('{encoded}'); const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}}); const url = URL.createObjectURL(blob);
const anchor = document.createElement('a'); anchor.href = url; anchor.download = '{zip_path.name}';
document.body.appendChild(anchor); anchor.click(); anchor.remove(); setTimeout(() => URL.revokeObjectURL(url), 60000);
"""))
print('Download avviato:', zip_path, f'({zip_path.stat().st_size / 2**20:.1f} MiB)', '| checkpoint inclusi:', include_checkpoints)